In [25]:
from pathlib import Path

import pandas as pd

FILENAME = "vabp_values_10k.parquet"

parquet_path = Path("hf_upload") / FILENAME
df = pd.read_parquet(parquet_path)
print(f"Loaded {len(df):,} rows from {parquet_path}")

# print(df.info())
print(df.describe(include='all'))
# print(df.head())

Loaded 10,000 rows from hf_upload\vabp_values_10k.parquet
                                      general_instruction  \
count                                               10000   
unique                                                  1   
top     You are a chess grandmaster currently playing ...   
freq                                                10000   
mean                                                  NaN   
std                                                   NaN   
min                                                   NaN   
25%                                                   NaN   
50%                                                   NaN   
75%                                                   NaN   
max                                                   NaN   

                                                 question  \
count                                               10000   
unique                                               7786   
top     Here is a board in

In [18]:
row = df.sample(1).iloc[0]

print("general_instruction:\n")
print(row["general_instruction"])
print("\n" + "=" * 100 + "\n")

print("question:\n")
print(row["question"])
print("\n" + "=" * 100 + "\n")

print("response:\n")
print(row["response"])
print("\n" + "=" * 100 + "\n")

print("fen_board:\n")
print(row["fen_board"])

general_instruction:

You are a helpful chess assistant.


question:

Here is a board in a game you're currently playing:
8| . . . . . r k .
7| . r . . b p p .
6| p . . . p . . p
5| B . . . P . . .
4| . p . R . P . .
3| . . . . . . . .
2| P P P R . . P P
1| . . K . . . . .
   _ _ _ _ _ _ _ _
   A B C D E F G H

- It is Black’s turn to move.
- No castling rights available.
- No en passant target square.
- Halfmove clock: 6
- Fullmove number: 27

Answer the following - if multiple questions, include a space between each answer:
What are the likely next 4 plies that would play out? List all moves in UCI notation (e.g., e4f6) separated by spaces. If a checkmate occurs, end with 'mate'. Otherwise, end with the change in centipawns in the format '[Δ+/-#]'.


response:

g7g5 c2c3 b4c3 a5c3 [Δ+56]


fen_board:

5rk1/1r2bpp1/p3p2p/B3P3/1p1R1P2/8/PPPR2PP/2K5 b - - 6 27


In [19]:
import re


def _strip_bracketed_suffix(value):
    if pd.isna(value):
        return value
    text = str(value)
    return re.sub(r"\s*\[[^\]]*\]\s*$", "", text).strip()


def overwrite_parquet_with_dedup(
    parquet_path,
    subset_columns,
    normalizers=None,
):
    parquet_path = Path(parquet_path)
    frame = pd.read_parquet(parquet_path).copy()

    if isinstance(subset_columns, str):
        subset_columns = [subset_columns]

    missing_columns = [column for column in subset_columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"Missing columns for dedup: {missing_columns}")

    dedup_frame = frame.copy()
    normalizers = normalizers or {}
    dedup_subset = []

    for column in subset_columns:
        helper_column = column
        normalizer = normalizers.get(column)
        if normalizer is not None:
            helper_column = f"__dedup__{column}"
            dedup_frame[helper_column] = dedup_frame[column].map(normalizer)
        dedup_subset.append(helper_column)

    before = len(dedup_frame)
    deduped = dedup_frame.drop_duplicates(subset=dedup_subset).copy()
    helper_columns = [column for column in dedup_subset if column.startswith("__dedup__")]
    if helper_columns:
        deduped = deduped.drop(columns=helper_columns)

    deduped.to_parquet(parquet_path, index=False)
    print(
        {
            "parquet_path": str(parquet_path),
            "rows_before": before,
            "rows_after": len(deduped),
            "rows_removed": before - len(deduped),
            "subset_columns": list(subset_columns),
            "normalized_columns": sorted(normalizers),
        }
    )
    return deduped


# Example: drop rows whose response becomes identical after removing a trailing
# bracketed suffix such as '[\u0394+3]'. This overwrites the parquet in place.
deduped_df = overwrite_parquet_with_dedup(
    parquet_path,
    subset_columns=["response"],
    normalizers={"response": _strip_bracketed_suffix},
)
print(deduped_df.head())


{'parquet_path': 'hf_upload\\bestline_1p6mm.parquet', 'rows_before': 1614227, 'rows_after': 1604684, 'rows_removed': 9543, 'subset_columns': ['response'], 'normalized_columns': ['response']}
                  general_instruction  \
0  You are a helpful chess assistant.   
1  You are a helpful chess assistant.   
2  You are a helpful chess assistant.   
3  You are a helpful chess assistant.   
4  You are a helpful chess assistant.   

                                            question  \
0  Here is a board in a game you're currently pla...   
1  Here is a board in a game you're currently pla...   
2  Here is a board in a game you're currently pla...   
3  Here is a board in a game you're currently pla...   
4  Here is a board in a game you're currently pla...   

                         response  \
0  f3d2 a7a5 b2c3 a5b4 a3b4 [Δ-8]   
1       c5e7 a5a6 c3d2 a6e6 [Δ+4]   
2      f7e6 f1f7 d8d7 c1f4 [Δ+32]   
3  a6b5 d4b2 g7f7 b2a3 b5b4 [Δ+8]   
4      f3f4 h2g4 f4g5 d5d4 [Δ+79]   

  